In [ ]:
import numpy as np
import scipy.stats as stats

from main_analyses.dual_pfc_funcs import load_dict, getParams

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.style.use('scifigs.mplstyle')
SAVE_FIG = False

# parameters
params = getParams()
subjects = params['subjects']
plot_sym = params['markers']
color_map = params['color_map']

In [ ]:
CROSSVAL = False
data_path = 'preprocessed_data/'
if CROSSVAL:
    dat = load_dict(data_path + 'pupil_prediction_cv.pkl')
else:
    dat = load_dict(data_path + 'pupil_prediction.pkl')
fnames = dat.keys()

In [ ]:
# example session pupil prediction
ex_dat = dat['Sa191206']
y = ex_dat['pupil_zsc']
print('Total trials in example session: N={}'.format(y.shape[0]))

# predicitons
yhat_across = ex_dat['predictions']['across']
yhat_within_left = ex_dat['predictions']['within-left']
yhat_within_right = ex_dat['predictions']['within-right']

print('R2 values - left {:.5f}, right {:.5f}, across {:.5f}'.format(ex_dat['r2']['within-left'], ex_dat['r2']['within-right'], ex_dat['r2']['across']))

fig,ax = plt.subplots(1,1)
fig.set_figheight(4)

start,end = 300,340
idx = np.arange(0,end-start,1)
ax.plot(idx,y[start:end],'k',label='Actual pupil')
ax.plot(idx,yhat_across[start:end],label='Across-area regression', color=color_map['across'])
ax.plot(idx,yhat_within_left[start:end],label='Within-area (left) regression', color=color_map['within2'])
ax.plot(idx,yhat_within_right[start:end],label='Within-area (right) regression', color=color_map['within1'])

ax.legend(loc='upper right')
ax.set_xlabel('trial number (example session)')
ax.set_xticks(np.arange(0,end-start+1,10))
ax.set_xlim(-0.5,end-start+1)
ax.set_ylabel('trial-by-trial pupil diameter\n(normalized)')
ax.set_yticks(ticks=np.arange(-1,5,1))

if SAVE_FIG:
    pdf = PdfPages('figs/pupil_pred_ex_sess.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
col = 'k'
fig,ax = plt.subplots()
for i,latent in enumerate(['across','within-right','within-left']):
    for j,sub in enumerate(subjects):
        # trial-by-trial pupil
        curr = np.array([dat[sess]['r2'][latent] for sess in fnames if sess[:2]==sub[:2].title()])
        if i==0:
            ax.errorbar(i*3+j*.7,curr.mean(),yerr=stats.sem(curr),color=col,marker=plot_sym[j],ms=6,label=f'Monkey {sub[:2].title()}')
        else:
            ax.errorbar(i*3+j*.7,curr.mean(),yerr=stats.sem(curr),color=col,marker=plot_sym[j],ms=6)
        
        # null dist
        null = np.array([dat[sess]['null_r2'][latent] for sess in fnames if sess[:2]==sub[:2].title()]).flatten()
        null_prc = np.percentile(null,[2.5,97.5])
        ax.plot([i*3+j*.7,i*3+j*.7],null_prc,'k-',alpha=0.4,linewidth=3)
        
# formatting
ax.set_xticks(ticks=[0.7,3.7,6.7])
ax.set_yticks(ticks=np.arange(0,0.4,0.1))
ax.set_xticklabels(['across-area','within-area \n (right)','within-area \n (left)'])
ax.legend()
ax.set_ylabel('trial-by-trial pupilvdiameter \n variance explained ($r^2$)')

colors = [color_map['across'],color_map['within1'],color_map['within2']]
for xtick, color in zip(ax.get_xticklabels(), colors):
    xtick.set_color(color)

if SAVE_FIG:
    pdf = PdfPages('figs/pupil_pred_all_sess.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# pooled statistics
alpha = 0.01
print("pooled r2 statistics:")
all_diffs = np.array([])
for j,sub in enumerate(subjects):
    across = np.array([dat[sess]['r2']['across'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_left = np.array([dat[sess]['r2']['within-left'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_right = np.array([dat[sess]['r2']['within-right'] for sess in fnames if sess[:2]==sub[:2].title()])
    diffs = np.concatenate([across - within_left, across - within_right]) # across - within
    
    # combine across subjects
    all_diffs = np.concatenate((all_diffs, diffs))

    p = stats.ttest_1samp(diffs, popmean=0, alternative='greater').pvalue
    print(f'  {sub} across > within, p={p:.1E}')

# pooled:
print()
print('Pooled across subjects t-test')
p = stats.ttest_1samp(all_diffs, popmean=0, alternative='greater').pvalue
print(f'  across > within, p={p:.1E}')

In [ ]:
# repeat analysis with 1D predictions:
if CROSSVAL:
    control_dat = load_dict(data_path + 'pupil_prediction_1d_cv.pkl')
else:
    control_dat = load_dict(data_path + 'pupil_prediction_1d.pkl')
fnames = control_dat.keys()

# Statistics:
print('Trial-by-trial control')
all_across, all_left, all_right = np.array([]),np.array([]),np.array([])
for j,sub in enumerate(subjects):
    across = np.array([control_dat[sess]['r2']['across'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_left = np.array([control_dat[sess]['r2']['within-left'] for sess in fnames if sess[:2]==sub[:2].title()])
    within_right = np.array([control_dat[sess]['r2']['within-right'] for sess in fnames if sess[:2]==sub[:2].title()])
        
    # combine across areas
    all_across = np.concatenate((all_across, across))
    all_left = np.concatenate((all_left, within_left))
    all_right = np.concatenate((all_right, within_right))

# pooled:
print('Average r2 values: across {:.3f}, within left {:.3f}, within right {:.3f}'.format(all_across.mean(), all_left.mean(), all_right.mean()))

# pooled statistics across brain areas
r2_diffs = np.concatenate([all_across - all_left, all_across - all_right]) # across - within
p = stats.ttest_1samp(r2_diffs, popmean=0, alternative='greater').pvalue
print(f'pooled across > within, p={p:.1E}')